# 01 — EDA and data quality

Work locally on the IBM Telco Customer Churn CSV. Goal: understand the label, find data issues, and decide which columns are legal features.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

## Target balance

About 26.5% of customers churned. Accuracy is a weak headline metric because always predicting "No" is already ~73.5% accurate.

In [ ]:
df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True)

## TotalCharges is not actually numeric

Blank strings appear for brand-new customers (`tenure == 0`).

In [ ]:
blank_total = df["TotalCharges"].astype(str).str.strip().eq("")
print("blank TotalCharges rows:", int(blank_total.sum()))
df.loc[blank_total, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
clean = df.copy()
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"], errors="coerce")
print(clean["TotalCharges"].isna().sum())
clean["TotalCharges"] = clean["TotalCharges"].fillna(0)
clean["ChurnFlag"] = clean["Churn"].map({"Yes": 1, "No": 0})

## Business slices

Month-to-month contracts and electronic-check payers churn more. Long tenure churns less.

In [ ]:
clean.groupby("Contract")["ChurnFlag"].mean().sort_values(ascending=False)

In [ ]:
clean.groupby("PaymentMethod")["ChurnFlag"].mean().sort_values(ascending=False)

In [ ]:
clean.groupby(pd.cut(clean["tenure"], bins=[-0.1, 6, 12, 24, 48, 72]))["ChurnFlag"].mean()

## Leakage check

`customerID` is an identifier. Do not train on it. This file has no explicit churn-reason column, which is good.

In [ ]:
assert "ChurnReason" not in clean.columns
print("identifier unique:", clean["customerID"].nunique() == len(clean))
print("columns:", list(clean.columns))